# Primary readout

Conversion rate for the three pre-registered comparisons (Mens vs No E-Mail, Womens vs No E-Mail, Mens vs Womens), Holm corrected, followed by the spend readout. Methods are fixed in `docs/experiment_design.md` and `docs/Log.md` before this notebook was run:
- Wald (unpooled) CI as the main interval, Newcombe hybrid score as a cross-check
- CIs at alpha/3 (98.33%) as the main interval, 95% shown as context
- z-test uses the same unpooled SE as the CI, pooled SE shown once as a cross-check

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from scipy import stats
from load import load_hillstrom

df = load_hillstrom()
TABLES = Path.cwd().parent / "results" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

print(f"rows: {len(df)}")

rows: 64000


## 0. Pre-check, are there enough conversions for the Wald interval?

The Wald interval leans on a normal approximation, which gets shaky when an arm has few events. Before splitting any outcome by arm, checking how many conversions each arm should have using only pooled numbers:

expected conversions in an arm = pooled conversion rate x arm size

The pooled rate is the same 0.9031% validated in `00_eda_pre_registration.ipynb`. Arm sizes are counts of `segment`. Neither splits an outcome by arm.

In [2]:
pooled_rate = df["conversion"].mean()
total_conversions = df["conversion"].sum()
arm_sizes = df["segment"].value_counts().sort_index()
expected_conversions = pooled_rate * arm_sizes
expected_non_conversions = (1 - pooled_rate) * arm_sizes

print(f"pooled conversion rate: {pooled_rate:.4%}")
print(f"total conversions (all arms pooled): {total_conversions:,}")
print()
pd.DataFrame({
    "arm_size": arm_sizes,
    "expected_conversions": expected_conversions,
    "expected_non_conversions": expected_non_conversions,
})

pooled conversion rate: 0.9031%
total conversions (all arms pooled): 578



,arm_size,expected_conversions,expected_non_conversions
segment,,,
Mens E-Mail,21307,192.428844,21114.571156
No E-Mail,21306,192.419812,21113.580187
Womens E-Mail,21387,193.151344,21193.848656
